In [14]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
import os

In [15]:
load_dotenv()
ollama_api_key = os.getenv("OLLAMA_API_KEY")


In [16]:
model=ChatOllama(
    model="gemma4",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {"Authorization": f"Bearer {ollama_api_key}"}
    }
)

In [17]:
#create state
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation_score: str

In [18]:
#declare constants
CREATE_OUTLINE="create_outline"
CREATE_BLOG="create_blog"
EVALUATE_BLOG="evaluate_blog"

In [19]:
def create_outline(state:BlogState)->BlogState:
    #extract title from state
    title=state["title"]
    
    #form prompt
    prompt=f"Write an detailed outline for a blog on a topic {title}"
    
    #generate outline
    outline=model.invoke(prompt).content
    
    #update state
    state["outline"]=outline
    
    return state

In [20]:
def create_blog(state:BlogState)->BlogState:
    #extract outline from state
    title=state["title"]
    outline=state["outline"]
    
    #form prompt
    prompt=f"Write a detailed blog on a topic {title} using following outline\n {outline}"
    
    #generate blog content
    blog_content=model.invoke(prompt).content
    
    #update state
    state["content"]=blog_content
    
    return state

In [21]:
def evaluate_blog(state:BlogState)->BlogState:
    #extract content from state
    content=state["content"]
    outline=state["outline"]
    #form prompt
    prompt=f"Evaluate the blog based on outline {outline} and content\n{content}"
    
    #generate evaluation
    evaluation=model.invoke(prompt).content
    
    #update state
    state["evaluation_score"]=evaluation
    
    return state

In [22]:
graph=StateGraph(BlogState)
# add nodes to graph
graph.add_node(CREATE_OUTLINE,create_outline)
graph.add_node(CREATE_BLOG,create_blog)
graph.add_node(EVALUATE_BLOG,evaluate_blog)

#add edges to graph
graph.add_edge(START,CREATE_OUTLINE)
graph.add_edge(CREATE_OUTLINE,CREATE_BLOG)
graph.add_edge(CREATE_BLOG,EVALUATE_BLOG)

graph.add_edge(EVALUATE_BLOG,END)

In [23]:
# compile graph
workflow=graph.compile()

In [24]:
#initial state
initial_state={"title":"Rise of AI in India"}
final_state=workflow.invoke(initial_state)
print(final_state)

{'title': 'Rise of AI in India', 'outline': 'This is a detailed outline for a comprehensive blog post titled **"The Rise of AI in India: From Outsourcing Hub to Innovation Powerhouse."**\n\nThe goal of this blog is to position India not just as a consumer of AI, but as a global leader in AI development and implementation.\n\n---\n\n# Blog Outline: The Rise of AI in India\n\n## 1. Introduction\n*   **The Hook:** Contrast the "Old India" (the back-office/BPO capital of the world) with the "New India" (a hub for AI research, startups, and digital infrastructure).\n*   **The Current State:** Brief mention of the rapid adoption of Generative AI (ChatGPT, Gemini, etc.) across Indian demographics.\n*   **Thesis Statement:** India is uniquely positioned to lead the AI revolution due to its massive data pools, a young tech-savvy population, and a strategic shift from "service-led" to "product-led" growth.\n*   **What the reader will learn:** A roadmap of the blog (Economic impact, Government in

In [25]:
print(final_state["outline"])

This is a detailed outline for a comprehensive blog post titled **"The Rise of AI in India: From Outsourcing Hub to Innovation Powerhouse."**

The goal of this blog is to position India not just as a consumer of AI, but as a global leader in AI development and implementation.

---

# Blog Outline: The Rise of AI in India

## 1. Introduction
*   **The Hook:** Contrast the "Old India" (the back-office/BPO capital of the world) with the "New India" (a hub for AI research, startups, and digital infrastructure).
*   **The Current State:** Brief mention of the rapid adoption of Generative AI (ChatGPT, Gemini, etc.) across Indian demographics.
*   **Thesis Statement:** India is uniquely positioned to lead the AI revolution due to its massive data pools, a young tech-savvy population, and a strategic shift from "service-led" to "product-led" growth.
*   **What the reader will learn:** A roadmap of the blog (Economic impact, Government initiatives, Sectoral shifts, and Challenges).

## 2. The C

In [26]:
print(final_state["content"])

# The Rise of AI in India: From Outsourcing Hub to Innovation Powerhouse

For decades, the global narrative surrounding India’s tech industry was defined by the "Back Office." From the sprawling campuses of Bengaluru to the hubs of Hyderabad, India was seen as the world’s BPO (Business Process Outsourcing) capital—the place where the world sent its software maintenance, customer support, and legacy code to be managed.

But a quiet, powerful transformation has taken place.

The "Old India" of service-led growth is giving way to a "New India" of product-led innovation. With the explosion of Generative AI—from ChatGPT to Gemini—India is no longer just executing the world's blueprints; it is drafting its own. India is uniquely positioned to lead the AI revolution, leveraging massive data pools, a young tech-savvy population, and a strategic pivot toward high-value invention.

In this blog, we will explore how India is transitioning from a service provider to a global AI leader, the sectors

In [27]:
print(final_state["evaluation_score"])

### Evaluation Report: "The Rise of AI in India"

Overall, the execution of the blog is **excellent**. The writer adhered strictly to the provided outline, ensuring that every strategic point was covered while maintaining a professional, forward-looking, and authoritative tone.

Below is a detailed evaluation based on specific criteria:

---

#### 1. Adherence to Outline: **10/10**
The content follows the roadmap perfectly. 
*   **Introduction:** Successfully contrasts "Old India" vs. "New India."
*   **Catalysts:** Covers the Talent Pool, India Stack, and Demographics.
*   **Sectoral Shifts:** Provides concrete examples for AgriTech, HealthTech, FinTech, EdTech, and GovTech.
*   **Policy/Startups:** Properly references NITI Aayog, the IndiaAI Mission, and Bhashini.
*   **Challenges:** Honestly addresses job displacement and infrastructure gaps.
*   **Conclusion:** Ends with the predicted transition to "AI Invention" and a clear CTA.

#### 2. Tone and Narrative Flow: **9/10**
*   **Nar